# Somo la 03 - Mifumo ya Ubunifu ya Wakala

Katika somo hili, tunachunguza mifumo mitatu ya msingi ya ubunifu kwa ajili ya kujenga mawakala bora wa AI:

1. **Maelekezo ya Wakala Wazi** — Kutengeneza maagizo sahihi yanayoelezea jukumu yanayomwelekeza wakala
2. **Matokeo Yenye Muundo Kwa Modeli za Pydantic** — Kuhakikisha mawakala wanarudisha data inayotabirika na kuthibitishwa
3. **Mawakala wa Jukumu Moja Tu** — Kubuni mawakala waliolenga ambao kila mmoja hufanya jambo moja vizuri

Tuta tumia kila mfumo katika hali ya **mshauri wa mahali pa kusafiria**, tukijenga taratibu mfumo unaoweza kupendekeza maeneo, kuangalia upatikanaji, na kushughulikia mpangilio.


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mfano 1: Maelekezo Wazi kwa Wakala

Mfano wenye athari kubwa pia ni rahisi: kuandika maelekezo wazi, ya kina kwa wakala wako.

Maelekezo mazuri huainisha:
- **Nani** wakala ni (tabia na mtindo wa kuzungumza)
- **Nini** anapaswa kufanya (majukumu hatua kwa hatua)
- **Jinsi** anapaswa kuishi (vikwazo na mtindo)

Hapo chini, tunaunda wakala wa mtaalamu wa usafiri na maelekezo ya wazi ambayo huunda kila jibu anazotoa.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mfano 2: Matokeo Yalio Pangiliwa na Modeli za Pydantic

Maandishi ya aina huru ni muhimu kwa mazungumzo, lakini mifumo ya chini inahitaji data iliyo pangiliwa.
Kwa kuunganisha **modeli za Pydantic** na **kifaa cha kazi**, tunaweza:

- Eleza mfumo kamili kwa matokeo ya wakala
- Thibitisha majibu moja kwa moja
- Unganisha matokeo ya wakala katika mantiki ya programu kwa kuaminika

Jambo muhimu la utekelezaji ni kupitisha `response_format` tunapomkimbiza wakala. Hii inalazimisha
mfano kurudisha kitu cha `TravelRecommendations` kilichothibitishwa (kinapatikana kwenye `response.value`)
badala ya maandishi ya aina huru. Kifaa cha `get_destination_details` pia hurudisha
`DestinationRecommendation` kilicho na aina, hivyo data inabaki imepangiliwa kuanzia mwanzo hadi mwisho.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Muundo wa 3: Wakala wa Majukumu Moja

Kazi ngumu huhitaji kugawanyika kati ya wawakala wengi wenye lengo mahususi, kila mmoja akiwa na jukumu moja:

- **Mtaalamu wa Mahali** anayejua kuhusu maeneo na upatikanaji
- **Mpangaji wa Usafirishaji** anayesimamia ndege, hoteli, na ratiba

Hii inaendana na kanuni ya uhandisi wa programu ya *mgawanyiko wa majukumu* — kila wakala ni rahisi kujaribu, kudumisha, na kuboresha kwa kujitegemea.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Muhtasari

Katika somo hili tulitumia mifumo mitatu ya muundo wa mawakala katika hali ya kupendekeza safari:

| Mfano | Wazo Muhimu | Faida |
|---|---|---|
| **Maelekezo Wazi** | Eleza utu, majukumu, na vizingiti mapema | Tabia ya mwakala inayolingana, ya chapa |
| **Matokeo Yaliyojengwa Kwa Mpangilio** | Tumia mifano ya Pydantic kama muundo wa majibu | Matokeo yaliyoripotiwa kwa mashine, yaliyothibitishwa |
| **Jukumu Moja Tu** | Mpa kila mwakala kazi moja maalum | Rahisi kujaribu, kudumisha, na kuunganisha |

Mifano hii huunganishwa kwa asili — unaweza kuunganisha maelekezo wazi na matokeo yaliyojengwa ndani ya mwakala mwenye jukumu moja ili kujenga mifumo imara, tayari kwa uzalishaji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
